# 🛒 Product Query Agent with LangChain & Groq

In this notebook, we build an intelligent **Product Query Agent** using:
- **LangChain** (`create_agent`, `@tool`, messages)
- **Groq** (`ChatGroq` with high-speed LLMs like `openai/gpt-oss-120b` or `qwen/qwen3.8-27b`)
- **Custom Tools**: Search, Spec Details, Side-by-Side Comparison, Stock & Delivery check, and Promotional Discounts.

## 1. Setup & Imports

In [ ]:
import os
import sys
from dotenv import load_dotenv

# Ensure package path is included
sys.path.insert(0, "src")

# Load API Keys from .env
load_dotenv("src/agentic_ai/.env")
load_dotenv(".env")

print("Groq API Key Configured:", bool(os.getenv("GROQ_API_KEY")))

## 2. Initialize ChatGroq LLM

In [ ]:
from langchain_groq import ChatGroq

# Initialize ChatGroq with fast tool-calling model
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    max_retries=2
)

print("LLM Initialized:", llm.model_name)

## 3. Define Custom Product Tools
We import our 5 specialized tools (`search_products`, `get_product_details`, `compare_products`, `check_inventory_and_delivery`, `get_active_discounts`).

In [ ]:
from agentic_ai.tools import (
    search_products,
    get_product_details,
    compare_products,
    check_inventory_and_delivery,
    get_active_discounts,
    PRODUCT_TOOLS
)

print(f"Loaded {len(PRODUCT_TOOLS)} Tools:")
for t in PRODUCT_TOOLS:
    print(f" - {t.name}: {t.description[:80]}...")

## 4. Construct the Product Query Agent
We use LangChain's `create_agent` with our custom system prompt and tools.

In [ ]:
from langchain.agents import create_agent
from agentic_ai.product_query_agent import SYSTEM_PROMPT, create_product_agent, run_product_query

# Create the compiled agent graph
agent = create_product_agent(model_name="openai/gpt-oss-120b")
print("Agent successfully constructed!")

## 5. Test Queries & Tool Execution

### Example 1: Product Search with Budget Filter & Deals

In [ ]:
query_1 = "What laptops do you have under $1200, and what discounts can I use?"
response_1 = run_product_query(agent, query_1)

print("=== TOOLS TRIGGERED ===")
for tc in response_1["tool_calls"]:
    print(tc)

print("\n=== AGENT RESPONSE ===\n")
print(response_1["output"])

### Example 2: Side-by-Side Comparison

In [ ]:
query_2 = "Compare the Apple iPhone 16 Pro and Samsung Galaxy S25 Ultra on camera, battery, and price."
response_2 = run_product_query(agent, query_2)

print(response_2["output"])

### Example 3: Real-Time Stock & Shipping Check

In [ ]:
query_3 = "Is the Sony WH-1000XM5 in stock? How fast can it ship to ZIP 90210?"
response_3 = run_product_query(agent, query_3)

print(response_3["output"])

## 6. Multi-turn Conversational Memory
We can maintain conversation history across multiple turns.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

# Turn 1
q1 = "Do you have noise cancelling headphones under $400?"
res1 = run_product_query(agent, q1, chat_history=history)
print("User:", q1)
print("Agent:", res1["output"][:250], "...\n")

history.append(HumanMessage(content=q1))
history.append(AIMessage(content=res1["output"]))

# Turn 2 (Contextual follow-up)
q2 = "Which one of those has the longest battery life and what coupons apply?"
res2 = run_product_query(agent, q2, chat_history=history)
print("User:", q2)
print("Agent:", res2["output"])
